In [ ]:
from utils.init_db import wipe_db
from pipeline_orchestration import full_pipeline_run
import duckdb
from pathlib import Path
import pandas as pd
import os
flag = 0

In [ ]:

# Change the current working directory to the project root
if flag == 0:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent
    os.chdir(PROJECT_ROOT)
    print(f"Notebook working directory set to project root: {Path.cwd()}")
    flag += 1
else:
    print(f"Notebook working directory set to project root: {Path.cwd()}")

In [ ]:
# wipe all data to make sure we are starting with a clean slate.
wipe_db(wipe_data=True, wipe_layer="all")

In [ ]:
# Initial Pipeline run - single entry point to run the entire pipeline.
full_pipeline_run()

In [ ]:
# Now add more data to each of the csvs

# The new data has the data for each turbine for 2022/04/01, but with the following values affected by missing data or outliers:

    # timestamp                  turbine_id        reason
    # 2022-04-01 00:00:00        1                 missing
    # 2022-04-01 18:00:00        2                 missing
    # 2022-04-01 15:00:00        3                 outlier
    # 2022-04-01 06:00:00        6                 outlier
    # 2022-04-01 10:00:00        9                 missing
    # 2022-04-01 15:00:00        10                missing
    # 2022-04-01 06:00:00        15                missing
    # 2022-04-01 15:00:00        13                missing
    # 2022-04-01 15:00:00        11                outlier


for data_group in [1, 2, 3]:
    df_base = pd.read_csv(f"data/mock_extra/original/data_group_{data_group}.csv")
    df_extra = pd.read_csv(f"data/mock_extra/data_group_{data_group}_extra.csv")

    df_union = pd.concat([df_base, df_extra], ignore_index=True)

    # print(len(df_base), len(df_extra), len(df_union))
    # Save to a location
    df_union.to_csv(f"data/a_raw/data_group_{data_group}.csv", index=False)
print("NEW BATCH OF BAD DATA ADDED")


In [ ]:
# Check the turbine summary data

temp_con = duckdb.connect("db/windfarm.duckdb")
cleaned_data_df = temp_con.execute("SELECT * FROM 'data/c_silver/turbine_clean/*.parquet';").df()
summary_data_df = temp_con.execute("SELECT * FROM 'data/d_gold/turbine_summary/*.parquet';").df()

print(cleaned_data_df)
print(summary_data_df)
temp_con.close()

In [ ]:
# Re-run the pipeline to see the incremental loading work.
import time
time.sleep(2)  # Sleep for a bit to ensure file system has registered the new files
full_pipeline_run()

In [ ]:
# Check the turbine summary data

temp_con = duckdb.connect("db/windfarm.duckdb")
cleaned_data_df = temp_con.execute("SELECT * FROM 'data/c_silver/turbine_clean/*.parquet' where timestamp > '2022-03-31 23:59:59';").df()
summary_data_df = temp_con.execute("SELECT * FROM 'data/d_gold/turbine_summary/*.parquet' where date > '2022-03-31';").df()
dq_df = temp_con.execute("SELECT * FROM 'data/c_silver/silver_data_quality_report/*.parquet';").df()

# Notice how we now have data from the new upload and data quality flags from the new data
print(cleaned_data_df)
print(summary_data_df)
print(dq_df)
temp_con.close()

In [ ]:
dq_df